In [1]:
# Model
from pprint import pprint

from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://host.docker.internal:11434",
    keep_alive="12m",
    model="gemma3:1b",
    temperature=0.24,
)
pprint(llm)

/workspaces/monorepo/applications/affirmations/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


ChatOllama(model='gemma3:1b', temperature=0.24, keep_alive='12m', base_url='http://host.docker.internal:11434')


In [2]:
# Run Model
from pprint import pprint

hello_world = llm.invoke("Hello, world!")
hello_world.pretty_print()
pprint(hello_world.model_dump(exclude={"content"}))

================================== Ai Message ==================================

Hello there! 👋 It's good to be chatting with you. How can I help you today?
{'additional_kwargs': {},
 'id': 'lc_run--019cbe19-ffb0-7ed1-b898-2e1f4446b955-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'created_at': '2026-03-05T13:05:02.85684Z',
                       'done': True,
                       'done_reason': 'stop',
                       'eval_count': 22,
                       'eval_duration': 255222544,
                       'load_duration': 1510500375,
                       'logprobs': None,
                       'model': 'gemma3:1b',
                       'model_name': 'gemma3:1b',
                       'model_provider': 'ollama',
                       'prompt_eval_count': 13,
                       'prompt_eval_duration': 132460625,
                       'total_duration': 2001503583},
 'tool_calls': [],
 'type': 'ai',
 'usage_metadata': {'input_tokens': 13,
  

In [ ]:
# Tools
import trafilatura
from langchain_community.utilities import SearxSearchWrapper
from langchain_core.tools import tool

from src.subjects import TAROT_CARDS

searxng_wrapper = SearxSearchWrapper(searx_host="http://searxng:8080")


@tool
def searxng_search(query: str) -> str:
    """Search using SearxNG, a self-hosted metasearch engine aggregating
    results from various sources. Fetches and returns the full text content
    of the top result pages for comprehensive research.

    Args:
        query: The search terms to look up, phrased as a concise natural language query.
    """
    searxng_results = searxng_wrapper.results(query, num_results=12)
    trafilatura_results = []
    for i, result in enumerate(searxng_results, start=1):
        url = result.get("link", "")
        title = result.get("title", "").strip()
        text = trafilatura.extract(
            trafilatura.fetch_url(url),
            include_comments=False,
            include_tables=True,
            no_fallback=False,
        )
        heading = title if title else url
        trafilatura_results.append(
            f"## Search Result {i}: {heading}\n\n**Source:** [{url}]({url})\n\n{text}"
        )

    if not trafilatura_results:
        raise ValueError(f"No content could be extracted for query: {query}")

    return "\n\n---\n\n".join(trafilatura_results)


subject = TAROT_CARDS[0]
tool_results = searxng_search.invoke(
    f'"{subject["name"]}" {subject["category"]["name"]} meaning interpretation significance symbolism themes correspondences'
)
pprint(tool_results)

('## Search Result 1: The Strength Card in Tarot: A Complete Guide to its '
 'Symbolism and Meaning\n'
 '\n'
 '**Source:** '
 '[https://tarot.ac/blog/the-strength-card-in-tarot-a-complete-guide-to-its-symbolism-and-meaning](https://tarot.ac/blog/the-strength-card-in-tarot-a-complete-guide-to-its-symbolism-and-meaning)\n'
 '\n'
 'The Strength Card in Tarot: A Complete Guide to its Symbolism and Meaning\n'
 'Table of Contents\n'
 '- 1. What is the Strength card in the Rider-Waite Tarot deck\n'
 '- 2. The Numbering of Arcana: Strength as the VIII or XI Arcana\n'
 '- 3. The Symbolism of the Strength Arcana and Its Deep Meaning\n'
 '- 4. The Meaning of the Strength Arcana in the Upright Position\n'
 '- 5. Interpretation of the Strength Card in Reverse\n'
 '- 6. Strength as a Significator in a Tarot Spread\n'
 '- 7. The Strength Card as the Card of the Day\n'
 '- 8. Meditation on the Strength Arcana: Connecting to Energy\n'
 '- 9. Conclusion: Integrating the Energy of the Force into Everyday

In [8]:
# Agent
from pathlib import Path

import inflect
from langchain_core.messages import HumanMessage, ToolMessage

from src.subjects import TAROT_CARDS

p = inflect.engine()


def ingest_document(subject):
    tool_message = ToolMessage(
        content=searxng_search.invoke(
            f'"{subject["name"]}" {subject["category"]["name"]} meaning interpretation significance symbolism themes correspondences'
        ),
        tool_call_id="searxng_search",
    )
    tool_message.pretty_print()

    human_message = HumanMessage(
        content=f"""\
    Summarize the provided web search results into comprehensive reference document about the meaning of "{subject["name"]}" ({subject["category"]["name"]}).
    Target length 1600 words. Use only the web search results provided as the source of truth. Structure the document following this markdown template:

    ```markdown
    # {subject["name"]}

    ## Meaning

    Explain the meaning and interpretation of "{subject["name"]}". Ignore its history and origins, just focus on the interpretation.
    Describe its significance within the category of "{p.plural(subject["category"]["name"])}" and how it relates to other subjects within the category.

    ## Application

    Summarize how the themes of "{subject["name"]}" apply to different aspects of life and the world, like love, relationships, work, finances, health, and fortune.

    ## Symbolism

    Highlight any key symbols, imagery, and archetypes associated with "{subject["name"]}" and what they represent.

    ## Correspondences
    [Only include rows that have a correspondence to "{subject["name"]}"]

    | Category  | Subject | Notes |
    | --------- | ------- | ----- |
    | tarot | ... | ... |
    | lenormand | ... | ... |
    | number | ... | ... |
    | suit | ... | ... |
    | rank | ... | ... |
    | zodiac | ... | ... |
    | planet | ... | ... |
    | element | ... | ... |
    | cardinality | ... | ... |
    | polarity | ... | ... |
    | rune | ... | ... |
    | chakra | ... | ... |
    | color | ... | ... |
    | solfeggio | ... | ... |
    | sephirot | ... | ... |
    | hebrew letter  | ... | ... |
    | kabbalah world | ... | ... |
    | gemstone | ... | ... |
    | metal | ... | ... |
    | weekday | ... | ... |
    ```

    Structure the document following the provided markdown template including the target lengths; do not add additional sections.
    Use only the information from the web search results, do not use any prior knowledge or information that is not included in the web search results.
    """
    )
    human_message.pretty_print()

    agent_message = llm.invoke([tool_message, human_message])
    agent_message.pretty_print()

    output_directory = Path(f"../../output/{subject['category']['slug']}")
    output_directory.mkdir(parents=True, exist_ok=True)
    output_filename = f"{subject['order']}-{subject['slug']}.md"
    output_path = output_directory / output_filename
    output_path.write_text(agent_message.content, encoding="utf-8")
    print(f"\nWritten to: {output_path.resolve()}\n")


for subject in TAROT_CARDS[46:47]:
    ingest_document(subject)

================================= Tool Message =================================

## Search Result 1: 6 Types of Hormones That Exercise Affects - Cathe Friedrich

**Source:** [https://cathe.com/the-hormonal-symphony-of-exercise-6-types-of-hormones-that-exercise-affects/](https://cathe.com/the-hormonal-symphony-of-exercise-6-types-of-hormones-that-exercise-affects/)

You’ve just finished a heart-pounding, sweat-dripping workout. Your muscles are singing, your heart is thumping, and you’re basking in that post-exercise glow. But did you know that behind the scenes, your body is conducting a complex hormonal symphony? This isn’t just about burning calories or building muscle. It’s about the intricate dance of hormones that exercise sets into motion, transforming not just your body, but your mind too.
Let’s explore this fascinating world, and how physical activity stimulates the release of powerful chemical messengers like endorphins, growth hormone, and testosterone. These substances regu